# EduRoute AI: evaluation on a Colab GPU

**Before you start:** *Runtime → Change runtime type → T4 GPU*, then
*Runtime → Run all*. When step 2 asks, upload `colab/eduroute_colab_bundle.zip`.
At the end a zip downloads: copy its files **into** `evaluation\results\`
(do not replace the folder).

Step 0 chooses what to run. The defaults run the **complete GPU evaluation**,
about 2 hours on a T4. Keep the tab open; if the session disconnects, run all
again: the benchmark resumes from Google Drive and skips finished answers.

| Step | What | Default | Time on a T4 |
|---|---|---|---|
| 5b | End-to-end smoke test of every route | on | ~2 min |
| 6 | Image questions: Qwen2.5-VL 3B and 7B compared | on | ~12 min |
| 7 | System benchmark: **all 130 questions** × 5 strategies, NLI on | on | ~80-100 min |
| 9 | Router check on held-out questions (extended index) | on | ~1 min |
| 10 | Voice word error rate | on | ~3 min |
| 11 | Claim-labelling export | on | seconds |

## 0. Choose what to run

In [ ]:
RUN_SMOKE = True         # end-to-end check of every route before the long runs
RUN_IMAGES = True        # image questions
# Vision models to compare on the image questions. 7B (~6 GB) fits the T4 next to the
# 3B text model; it is the candidate fix for the 3B model misreading tables.
VISION_MODELS = ['qwen2.5vl:3b', 'qwen2.5vl:7b']
RUN_BENCHMARK = True     # system benchmark
SAMPLE = None            # questions per set; None = all 130 questions (the full run)
RUN_ROUTER = True        # router held-out check on the extended (36,367-passage) index
RUN_ASR = True           # voice WER; upload voice_eval/ to use real recordings instead
# Keep progress and results in Google Drive (MyDrive/eduroute_colab): a disconnect
# then loses nothing, and "Run all" resumes the benchmark. False = session only.
SAVE_TO_DRIVE = True

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Upload the project bundle

If the upload is slow, put the zip in Google Drive and use the commented lines instead.

In [ ]:
import os, zipfile
from google.colab import files

uploaded = files.upload()                      # choose eduroute_colab_bundle.zip
bundle = next(iter(uploaded))
# from google.colab import drive; drive.mount('/content/drive')
# bundle = '/content/drive/MyDrive/eduroute_colab_bundle.zip'

zipfile.ZipFile(bundle).extractall('/content')
os.chdir('/content/eduroute')
os.makedirs('evaluation/results', exist_ok=True)
print(sorted(os.listdir('.')))

DRIVE_DIR = None
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DIR = '/content/drive/MyDrive/eduroute_colab'
    os.makedirs(DRIVE_DIR, exist_ok=True)
    # Results of an earlier, interrupted session come back, so nothing is redone.
    !cp -n {DRIVE_DIR}/*.json* evaluation/results/ 2>/dev/null; true
    print('Progress is saved to', DRIVE_DIR)

def save_progress():
    if DRIVE_DIR:
        !cp -r evaluation/results/. {DRIVE_DIR}/ 2>/dev/null; true

## 3. Install dependencies

In [ ]:
!pip -q install sentence-transformers faiss-cpu python-dotenv truststore requests Pillow pymupdf
if RUN_ASR or RUN_SMOKE:
    !pip -q install faster-whisper av edge-tts gTTS

## 4. Start Ollama on the GPU and download the models

In [ ]:
!apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null

import subprocess, time, requests
env = {**os.environ, 'OLLAMA_CONTEXT_LENGTH': '8192', 'OLLAMA_KEEP_ALIVE': '2h',
       'OLLAMA_MAX_LOADED_MODELS': '2'}   # keep the text and vision models loaded together
server = subprocess.Popen(['ollama', 'serve'], env=env,
                          stdout=open('/content/ollama.log', 'w'), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        print('Ollama', requests.get('http://localhost:11434/api/version').json()['version']); break
    except Exception:
        time.sleep(1)

def installed():
    return [m['name'] for m in requests.get('http://localhost:11434/api/tags').json()['models']]

!ollama pull qwen2.5:3b 2>&1 | tail -1
if RUN_IMAGES:
    for vision_model in VISION_MODELS:
        for attempt in (1, 2):
            !ollama pull {vision_model} 2>&1 | tail -1
            if vision_model in installed():
                break
    VISION_MODELS = [m for m in VISION_MODELS if m in installed()]
print('Installed models:', installed())
assert any(n.startswith('qwen2.5:3b') for n in installed()), 'Text model qwen2.5:3b is missing'
if RUN_IMAGES and not VISION_MODELS:
    RUN_IMAGES = False
    print('WARNING: no vision model could be downloaded, so the image step is skipped.')

## 5. Settings, speed check and one question end to end

In [ ]:
os.environ.update({
    'EDUROUTE_RAG_VERSION': 'engineering_v1',
    'EDUROUTE_LLM_BASE_URL': 'http://localhost:11434/v1',
    'EDUROUTE_LLM_MODEL': 'qwen2.5:3b',
    'EDUROUTE_LLM_TIMEOUT': '600',
    'EDUROUTE_NLI_MODEL': 'cross-encoder/nli-deberta-v3-xsmall',
    'EDUROUTE_VISION_BASE_URL': 'http://localhost:11434/v1',
    'EDUROUTE_VISION_MODEL': 'qwen2.5vl:3b',
    'EDUROUTE_VISION_TIMEOUT': '600',
})
r = requests.post('http://localhost:11434/api/generate', json={
    'model': 'qwen2.5:3b', 'prompt': 'Define slip in an induction motor in one sentence.', 'stream': False}).json()
print(f"Text model: {r['eval_count'] / r['eval_duration'] * 1e9:.0f} tokens/s (expect tens; ~1 means no GPU)")

from src.pipeline import EduRoutePipeline
pipeline = EduRoutePipeline(log=False)
r = pipeline.ask('Explain the working of a three-phase induction motor.', answer_mode='5-mark')
print('workflow:', r.workflow, '| grounded:', r.grounded, '| %.1f s' % (r.telemetry['total_ms'] / 1000))
print('\n' + r.answer)
print('\nNotices:', r.notices)
if r.verification:
    for c in r.verification['claims']:
        print(f"  [{c['label']:<11}] nli={c.get('nli')} {c['text'][:90]}")

## 5b. Smoke test: every route with the real models

The same check as on a server (`deploy/smoke_test.py`): GPU, both models, a cited
knowledge-base answer, a general answer, an image question and voice, each within
a time limit. A FAIL here means fix it before the long runs.

In [ ]:
if RUN_SMOKE:
    !python -m deploy.smoke_test 2>&1 | grep -v -E "Batches|Loading weights|FutureWarning|torch.jit"
    save_progress()

## 6. Image questions (Qwen2.5-VL)

8 starter images with exactly known answers (truth table, block diagram,
signal plot, bar chart, equation, circuit, code, flowchart), plus any real
images added to `evaluation/image_eval/` before the bundle was made.

In [ ]:
if RUN_IMAGES:
    for vision_model in VISION_MODELS:      # one result file per model, to compare
        print(f'\n===== {vision_model} =====')
        try:  # warm the vision model: its first load can take a minute
            warm = requests.post('http://localhost:11434/api/generate',
                                 json={'model': vision_model, 'prompt': 'Say OK.', 'stream': False},
                                 timeout=600).json()
            print('Vision model:', warm.get('error') or 'loaded')
        except Exception as error:
            print('Vision warm-up failed:', error)
        !EDUROUTE_VISION_MODEL={vision_model} python -m evaluation.run_image_benchmark 2>&1 | grep -v -E "Batches|Loading weights|FutureWarning|torch.jit"
        save_progress()
else:
    print('Skipped (RUN_IMAGES = False)')

## 7. System benchmark

Seeded sample: 20 in-domain, 20 out-of-domain and 10 extra questions × 5
strategies = 250 answers, the same questions as every earlier run. If the
session disconnects, run this cell again: it resumes and skips finished answers.

In [ ]:
# In Drive, every finished answer is saved at once: a disconnect loses nothing.
run_name = 'full' if not SAMPLE else f'sample{SAMPLE}'
PARTIAL = (f'{DRIVE_DIR}/system_benchmark_partial_colab_{run_name}.jsonl' if DRIVE_DIR
           else f'evaluation/results/system_benchmark_partial_colab_{run_name}.jsonl')
open(PARTIAL, 'a').close()
if RUN_BENCHMARK:
    sample_arg = f'--sample {SAMPLE}' if SAMPLE else ''
    done = sum(1 for _ in open(PARTIAL))
    print(f'{done} answers already done (resuming)' if done else 'Starting a new run')
    !python -m evaluation.run_system_benchmark {sample_arg} --seed 7 --resume {PARTIAL} 2>&1 | grep -v -E "Batches|Loading weights|FutureWarning|torch.jit"
    save_progress()
else:
    print('Skipped (RUN_BENCHMARK = False)')

## 8. Results

In [ ]:
import glob, json
import pandas as pd

runs = sorted(glob.glob('evaluation/results/system_benchmark_2*.json'))
if runs:
    report = json.load(open(runs[-1]))
    print('LLM:', report['llm'], '| NLI:', report['config'].get('nli_model'), '| questions:', report['questions'])
    table = pd.DataFrame({s: v['all'] for s, v in report['summary'].items()}).T
    cols = ['runs', 'failures', 'median_latency_ms', 'mean_input_tokens', 'mean_output_tokens',
            'grounded_rate', 'verified_rate', 'revised_rate', 'mean_unsupported_claim_rate',
            'answers_with_citations', 'answers_with_contradicted_claims', 'answers_with_unconfirmed_figures']
    display(table[[c for c in cols if c in table]])
images = sorted(glob.glob('evaluation/results/image_benchmark_*.json'))
if images:
    image_report = json.load(open(images[-1]))
    print('Image benchmark:', image_report.get('summary') or image_report.get('reason'))

## 9. Router check on held-out questions

In [ ]:
if RUN_ROUTER:
    !python -m evaluation.check_router_heldout --version engineering_v1 2>&1 | grep -v -E "Batches|Loading weights|FutureWarning" | tee evaluation/results/router_heldout_colab.txt
    save_progress()
else:
    print('Skipped (RUN_ROUTER = False)')

## 10. Voice word error rate (optional)

With recordings, upload a folder `voice_eval/` (audio files plus
`references.csv` with columns `file,reference,speaker,accent`) and it is used
instead of the synthetic voices.

In [ ]:
if RUN_ASR:
    if os.path.exists('voice_eval/references.csv'):
        !python -m evaluation.evaluate_asr voice_eval 2>&1 | tail -6
    else:
        # the same 20 synthetic recordings as the laptop baseline (comparable WER)
        !python -m evaluation.evaluate_asr evaluation/voice_eval_synthetic 2>&1 | tail -6
else:
    print('Skipped (RUN_ASR = False)')

## 11. Export claims for human labelling

In [ ]:
if os.path.getsize(PARTIAL):
    !python -m evaluation.claim_labels export {PARTIAL} --n 120

## 12. Download the results

Copy the files from this zip **into** the project's `evaluation\results\`
folder (do not replace the folder). Keep `grading_key_*.json` away from the raters.

In [ ]:
import shutil
if not PARTIAL.startswith('evaluation/'):
    shutil.copy(PARTIAL, 'evaluation/results/')   # the per-answer file, kept in Drive
save_progress()
!cd evaluation && zip -q -r /content/eduroute_results.zip results
files.download('/content/eduroute_results.zip')